In [37]:
from cgra import *
from kernels import *
from sat_to_csv import *
import random


In [38]:
kernel_name = "relu"
version = "_meth"

In [39]:
# Global variables
CGRB_N_ROWS = 4
CGRA_N_COLS = 4
# Adress
first_addr = 20000

In [40]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [41]:
# Data
def configMemory(data, data_len):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------
    # &Im           -         &Im         -
    # nIt  

    nIterations = int(data_len/16)

    config_vals_col0 = [first_addr, nIterations]
    config_vals_col1 = []
    config_vals_col2 = [first_addr]
    config_vals_col3 = []
    
    addr_config_loads_col0 = 0
    kernel_add_memory_region(kernel_name, addr_config_loads_col0, config_vals_col0, version=version)
    addr_config_loads_col1 = addr_config_loads_col0 + len(config_vals_col0)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col1, config_vals_col1, version=version)
    addr_config_loads_col2 = addr_config_loads_col1 + len(config_vals_col1)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col2, config_vals_col2, version=version)
    addr_config_loads_col3 = addr_config_loads_col2 + len(config_vals_col2)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col3, config_vals_col3, version=version)
    # Load data
    kernel_add_memory_region(kernel_name, first_addr, data, version=version)
    # Config data address for direct loads
    load_addrs = [addr_config_loads_col0, addr_config_loads_col1, addr_config_loads_col2, addr_config_loads_col3]
    return load_addrs

In [42]:
def runKernel(load_addrs, max_it=1000):
    # Run kernel
    run(kernel_name, pr=["ROUT","INST"], load_addrs=load_addrs, version=version, limit=max_it)

In [43]:
def getResult(first_addr_C, end_addr_C, vlen):
    result = [0 for _ in range(vlen)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [44]:
def relu_cpu(image, vlen):
    expected_res = [0 for _ in range(vlen)]
    for i in range(vlen):
        if image[i] > 0:
            expected_res[i] = image[i]
        else:
            expected_res[i] = 0
    return expected_res

In [45]:
# Test dimensions
IMAGE_SIZE = 16*20 # Multiplo de 16
image = [random.randint(-20, 20) for _ in range(IMAGE_SIZE)]


load_addrs = configMemory(image, IMAGE_SIZE)

In [46]:
runKernel(load_addrs, max_it=200000)

Instr =  0 ( 0 )
[20000,    0,    0,    0]    [LWD R0  4, NOP , NOP , NOP ]    
[   0,    0,    0,    0]    [NOP , NOP , NOP , NOP ]    
[   0,    0, 20000,    0]    [NOP , NOP , LWD R0  4, NOP ]    
[   0,    0,    0,    0]    [NOP , NOP , NOP , NOP ]    
-------
Instr =  1 ( 1 )
[  20, 20004,    0, 20012]    [LWD R1  4, SADD R0  RCL  4, NOP , SADD R0  RCR  12]    
[20016,    0, 20024,    0]    [SADD R0  RCT  16, NOP , SADD R0  RCB  24, NOP ]    
[   0, 20036, 20040, 20044]    [NOP , SADD R0  RCR  36, SADD R0  R0  40, SADD R0  RCL  44]    
[20048,    0, 20056,    0]    [SADD R0  RCB  48, NOP , SADD R0  RCT  56, NOP ]    
-------
Instr =  2 ( 2 )
[   0, 20004, 20008, 20012]    [SADD R2  ZERO  ZERO, NOP , SADD R0  RCL  4, NOP ]    
[20016, 20020, 20024, 20028]    [NOP , SADD R0  RCL  4, NOP , SADD R0  RCL  4]    
[20032, 20036, 20040, 20044]    [SADD R0  RCT  16, NOP , NOP , NOP ]    
[20048, 20052, 20056, 20060]    [NOP , SADD R0  RCB  48, NOP , SADD R0  RCB  48]    
-------
Instr =  3

In [47]:
# Get result from CGRA
end_addr = first_addr + IMAGE_SIZE*4
result = getResult(first_addr, end_addr, IMAGE_SIZE)

# Get cpu output
expected_res = relu_cpu(image, IMAGE_SIZE)

# Check result correctness
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
if errors > 0:
    print("Err: " + str(errors))
    print("CGRA: " + str(result))
    print("CPU:  " + str(expected_res))
else:
    print("OK")



OK
